In [16]:
import numpy as np
import time

# DETERMINANTES

### Análisis de Complejidad

* **Complejidad Temporal:**
    *  $\mathcal{O}(n^3)$ . NumPy utiliza el método de descomposición LU (similar a la eliminación de Gauss) para calcular el determinante. Esto hace que el cálculo sea extremadamente rápido incluso con matrices grandes.


In [74]:
tamaño = int(input('Ingrese el tamaño de la matriz'))

def determinante(tamaño):
    matriz = np.random.randint(1,100, size=(tamaño,tamaño))
    print(f'Matriz de {tamaño} * {tamaño}:\n{matriz} ')
    determinante = np.linalg.det(matriz)
    return f'Su determinante es: {determinante}'

inicio = time.time()
var = determinante(tamaño)
fin= time.time()
print(f'Tiempo de cálculo: {round(fin - inicio, 4)} segundos')


Ingrese el tamaño de la matriz 3


Matriz de 3 * 3:
[[42 57 18]
 [15 48 37]
 [91 73 83]] 
Tiempo de cálculo: 0.0006 segundos


# PERMANENTE: Manera 1

### Análisis de Complejidad (Permanente con Permutaciones)

* **Complejidad Temporal:**
    * $\mathcal{O}(n \cdot n!)$ . El código genera todas las permutaciones posibles de las columnas ($n!$) y, por cada una, recorre las filas ($n$) para multiplicar los elementos. Es un algoritmo extremadamente lento para matrices mayores a 10x10.



In [82]:
def calcular_permanente(matriz):
    n = len(matriz)
    permanente = 0
    # itertools.permutations genera las n! combinaciones de columnas
    for p in itertools.permutations(range(n)):
        producto_diagonal = 1
        for i in range(n):
            producto_diagonal *= matriz[i][p[i]]
        permanente += producto_diagonal
    return permanente

n = int(input('Ingrese el tamaño para la matriz aleatoria n*n: '))

if n > 10:
    print("¡Cuidado! El cálculo del permanente para matrices mayores a 10 puede tardar mucho o congelar tu PC.")
    confirmar = input("¿Deseas continuar de todos modos? (s/n): ")
    if confirmar.lower() != 's':
        n = 10
        print("Ajustado automáticamente a un tamaño seguro de 10x10.")

matriz_aleatoria = np.random.randint(1, 11, size=(n, n))
print(f'\nMatriz Aleatoria de {n} * {n}:\n{matriz_aleatoria}\n')

# 3. Calcular el permanente midiendo el tiempo de ejecución
inicio = time.time()
resultado_permanente = calcular_permanente(matriz_aleatoria)
fin = time.time()

print(f'El permanente es: {resultado_permanente}')
print(f'Tiempo de cálculo: {round(fin - inicio, 4)} segundos')


Ingrese el tamaño para la matriz aleatoria n*n:  60


¡Cuidado! El cálculo del permanente para matrices mayores a 10 puede tardar mucho o congelar tu PC.


¿Deseas continuar de todos modos? (s/n):  y


Ajustado automáticamente a un tamaño seguro de 10x10.

Matriz Aleatoria de 10 * 10:
[[10  1  2  1  2  5  2  6  8  3]
 [ 7  2 10  9  5  8  5  2  7  8]
 [ 1  5  7  3  4  5  7  3  4  4]
 [ 5  7  8  2  5 10  4  3  2  2]
 [10  7  1  4 10  7  4  6  5  6]
 [ 8  3  9  1  6  4  9  9  9  9]
 [ 9  6  2  9  2  2  8  9  7  2]
 [ 9  2  2  3  3  8  6 10  4  9]
 [ 4  5  8  5  3 10  5  7  9  9]
 [ 4  3  2 10  6  1 10  7  8  7]]

El permanente es: 90218311593150
Tiempo de cálculo: 4.8045 segundos


# PERMANENTE: Manera 2 

### Análisis de Complejidad (Fórmula de Ryser)

* **Complejidad Temporal:**
    * $\mathcal{O}(n \cdot 2^n)$ (Exponencial). El algoritmo recorre todas las combinaciones posibles de subconjuntos de columnas ($2^n$). En cada iteración, realiza operaciones vectoriales con NumPy (`np.sum` y `np.prod`) sobre las filas de la matriz ($n$). Es drásticamente más rápido que el método factorial, permitiendo calcular matrices de hasta 15x15 en pocos segundos.


In [70]:
import itertools

In [25]:

def ryser_permanent(M):
    """
    Calcula el permanente de una matriz usando la fórmula de Ryser.
    Complejidad de tiempo: O(n * 2^n)
    """
    n = M.shape[0]
    perm = 0
    
    # Precalculamos las potencias de 2 para las operaciones de bits
    potencias_dos = 1 << np.arange(n)
    
    # Iteramos sobre los 2^n - 1 subconjuntos posibles (excluyendo el conjunto vacío)
    for i in range(1, 2**n):
        # Creamos una máscara booleana de las columnas activas en esta iteración
        mask = (i & potencias_dos) > 0
        
        # Sumamos las columnas seleccionadas por cada fila
        row_sums = np.sum(M[:, mask], axis=1)
        
        # Multiplicamos el resultado de las sumas de todas las filas
        term = np.prod(row_sums)
        
        # Calculamos el signo: (-1)^(n - número de columnas seleccionadas)
        sign = (-1)**(n - mask.sum())
        
        perm += sign * term
        
    return perm

def calcular_permanente_aleatorio():
    try:
        n = int(input("Ingrese el tamaño de la matriz cuadrada (ej. 3 para una de 3x3): "))
        if n <= 0:
            print("El tamaño de la matriz debe ser un número entero positivo mayor a 0.")
            return
            
        # Advertencia de seguridad algorítmica
        if n > 15:
            print(f"\n¡Advertencia! El cálculo del permanente para una matriz de {n}x{n} es exponencial.")
            print(f"La computadora tendrá que realizar {2**n - 1} iteraciones complejas.")
            confirmacion = input("¿Deseas continuar de todos modos? (s/n): ")
            if confirmacion.lower() != 's':
                print("Ejecución cancelada.")
                return

        # Generar una matriz cuadrada aleatoria (del 1 al 10)
        matriz = np.random.randint(1, 11, size=(n, n))

        # Iniciar el temporizador
        tiempo_inicio = time.time()

        # Calcular el permanente usando nuestra implementación
        permanente = ryser_permanent(matriz)

        # Detener el temporizador
        tiempo_fin = time.time()
        tiempo_total = tiempo_fin - tiempo_inicio

        # Mostrar los resultados
        print("\n--- Resultados ---")
        print(f"Matriz generada ({n}x{n}):")
        print(matriz)
        print(f"\nPermanente de la matriz: {permanente:g}") 
        print(f"Tiempo de ejecución: {tiempo_total:.6f} segundos")

    except ValueError:
        print("Entrada no válida. Por favor, ingrese un número entero.")

# Ejecutar la función
calcular_permanente_aleatorio()

Ingrese el tamaño de la matriz cuadrada (ej. 3 para una de 3x3):  3



--- Resultados ---
Matriz generada (3x3):
[[2 8 9]
 [2 2 1]
 [4 3 4]]

Permanente de la matriz: 244
Tiempo de ejecución: 0.000436 segundos


# Algoritmo de la mochila 
### - Complejidad 

- La complejidad temporal del algoritmo es **O(n)**, donde n es la cantidad de objetos.


In [108]:
def back(pesos, valores, capacidad):
    valor_total = 0
    peso_total = 0
    val =[]
    for i in range(len(pesos)):
        if peso_total + pesos[i] <= capacidad:
            peso_total += pesos[i]
            valor_total += valores[i]
            val.append(valores[i])
    return f'Valores {valor_total} Valores {val} Pesos {peso_total}'

pesos = [2, 3, 4]
valores = [10,20,15]
capacidad = 5

back(pesos, valores, capacidad)


'Valores 30 Valores [10, 20] Pesos 5'